# Manual Scratch Training and Official Model Review

이 노트북은 사람이 직접 실행하는 실험용 노트북입니다. `runs/`와 `metrics/metrics_summary.csv`에는 쓰지 않으며, 학습 셀의 결과는 오직 `scratch/<SCRATCH_NAME>/`에 저장됩니다.

- `OFFICIAL_RUN_VERSION`: 이미 만들어진 공식 모델을 읽기 전용으로 확인합니다.
- `SCENARIO_SPLIT = True`: scenario 단위 train/test 분리를 사용합니다.
- `TRAIN_DATASET_VERSIONS`, `TEST_DATASET_VERSIONS`: scenario split에 사용할 dataset 버전을 직접 고릅니다.
- `DATASET_VERSIONS`: `SCENARIO_SPLIT = False`인 기존 랜덤 split 실험에서만 사용합니다.
- `FEATURE_COLUMNS_OVERRIDE`: `None`이면 정식 config의 피처 정책을 사용하고, 목록을 적으면 이 실험에서만 해당 피처를 사용합니다.

DVC에서 과거 상태의 파일을 확인하려면 해당 `features_vXXXXXX.csv` 또는 `runs/run_vXXXXXX` 파일이 현재 작업 폴더에 checkout 되어 있어야 합니다.

In [22]:
from pathlib import Path
import sys
import pandas as pd

# Jupyter를 프로젝트 루트 또는 notebook/ 폴더에서 시작해도 pipeline import가 됩니다.
PROJECT_ROOT = next(path for path in (Path.cwd(), Path.cwd().parent) if (path / 'pipeline').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.training import load_official_model, train_scratch_model, sample_class_ratio

# 여기만 직접 바꿔서 사용하세요.
OFFICIAL_RUN_VERSION = 'v000004'  # 기존 모델 검토만 할 때 사용, 필요 없으면 None
SCENARIO_SPLIT = True               # True이면 랜덤 split 없이 scenario 단위로 분리
TRAIN_DATASET_VERSIONS = ['v000001']  # scenario A 전체 train
TEST_DATASET_VERSIONS = ['v000002']   # scenario B 전체 test
DATASET_VERSIONS = ['v000002']      # False일 때만 사용: 기존 랜덤 split 실험
SCRATCH_NAME = 'scenario_split_try_001'  # scratch/ 아래 결과 폴더명
FEATURE_COLUMNS_OVERRIDE = None   # 예: ['frame_len', 'dns_id', 'ttl_max']
OVERWRITE_SCRATCH = False          # 같은 SCRATCH_NAME 결과를 교체할 때만 True
CONFIG_PATH = PROJECT_ROOT / 'config' / 'randomforest.yaml'

print('Project root:', PROJECT_ROOT)
print('Evaluation:', 'scenario split' if SCENARIO_SPLIT else 'legacy random split')
print('Datasets for scratch:', (TRAIN_DATASET_VERSIONS, TEST_DATASET_VERSIONS) if SCENARIO_SPLIT else DATASET_VERSIONS)
print('Scratch output:', PROJECT_ROOT / 'scratch' / SCRATCH_NAME)

Project root: /Users/yes/ML_teamproject
Datasets for scratch: ['v000002']
Scratch output: /Users/yes/ML_teamproject/scratch/manual_try_001


## 기존 공식 모델 읽기 전용 확인

아래 셀은 `runs/run_vXXXXXX/randomforest_vXXXXXX.joblib`을 읽기만 하며 파일을 생성하거나 변경하지 않습니다.

In [29]:
if OFFICIAL_RUN_VERSION is not None:
    official_artifact = load_official_model(OFFICIAL_RUN_VERSION, project_root=PROJECT_ROOT)
    official_metrics = official_artifact['metrics']
    display(pd.Series({
        'run_version': official_artifact['run_version'],
        'dataset_versions': official_artifact['dataset_versions'],
        'feature_count': len(official_artifact['feature_columns']),
        'accuracy': official_metrics['accuracy'],
        'precision': official_metrics['precision'],
        'recall': official_metrics['recall'],
        'f1_score': official_metrics['f1_score'],
        'roc_auc': official_metrics['roc_auc'],
    }, name='official model'))
else:
    print('OFFICIAL_RUN_VERSION is None: official model review skipped.')

run_version                    v000004
dataset_versions    [v000001, v000002]
feature_count                       48
accuracy                           1.0
precision                          1.0
recall                             1.0
f1_score                           1.0
roc_auc                            1.0
Name: official model, dtype: object

## Scratch 학습 실행

아래 셀은 모델을 새로 학습합니다. 산출물은 `scratch/<SCRATCH_NAME>/`에만 생성되고 공식 run 이력에는 포함되지 않습니다. 결과가 마음에 들면 같은 feature/config 선택으로 `main.py train`을 실행하여 공식 run으로 확정하세요.

In [26]:
scratch_result = train_scratch_model(
    dataset_versions=None if SCENARIO_SPLIT else DATASET_VERSIONS,
    train_dataset_versions=TRAIN_DATASET_VERSIONS if SCENARIO_SPLIT else None,
    test_dataset_versions=TEST_DATASET_VERSIONS if SCENARIO_SPLIT else None,
    scratch_name=SCRATCH_NAME,
    project_root=PROJECT_ROOT,
    config_path=CONFIG_PATH,
    feature_columns=FEATURE_COLUMNS_OVERRIDE,
    overwrite=OVERWRITE_SCRATCH,
)
scratch_metrics = scratch_result['metrics']
display(pd.Series({
    'scratch_name': scratch_metrics['scratch_name'],
    'evaluation_strategy': scratch_metrics['evaluation_strategy'],
    'train_dataset_versions': scratch_metrics['train_dataset_versions'],
    'test_dataset_versions': scratch_metrics['test_dataset_versions'],
    'dataset_versions': scratch_metrics['dataset_versions'],
    'class_ratio': scratch_metrics['class_ratio'],
    'feature_count': len(scratch_metrics['feature_columns']),
    'accuracy': scratch_metrics['accuracy'],
    'precision': scratch_metrics['precision'],
    'recall': scratch_metrics['recall'],
    'f1_score': scratch_metrics['f1_score'],
    'roc_auc': scratch_metrics['roc_auc'],
}, name='scratch training'))
print('Saved scratch files in:', scratch_result['paths']['run_dir'])

FileExistsError: Training output already exists: /Users/yes/ML_teamproject/scratch/manual_try_001/randomforest_scratch.joblib

In [31]:
# =========================
# 6. Feature importance
# =========================

artifact = scratch_result  # train_scratch_model()의 반환값
feature_columns = artifact["metrics"]["feature_columns"]
model = artifact["paths"]["model"]

import joblib
saved_artifact = joblib.load(model)
rf_model = saved_artifact["model"].named_steps["model"]

fi = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print("Top 30 feature importances:")
display(fi.head(30))

print("Features with non-zero importance:")
display(fi[fi["importance"] > 0])

Top 30 feature importances:


,feature,importance
39,query_entropy,0.128864
2,ip_len,0.119899
1,frame_cap_len,0.094195
37,query_name_len,0.090909
5,udp_length,0.088048
33,ttl_min,0.078225
7,src_port,0.074622
35,ttl_mean,0.069099
34,ttl_max,0.069004
0,frame_len,0.053941


Features with non-zero importance:


,feature,importance
39,query_entropy,1.288637e-01
2,ip_len,1.198990e-01
1,frame_cap_len,9.419473e-02
37,query_name_len,9.090886e-02
5,udp_length,8.804808e-02
33,ttl_min,7.822526e-02
7,src_port,7.462190e-02
35,ttl_mean,6.909863e-02
34,ttl_max,6.900408e-02
0,frame_len,5.394088e-02


In [20]:
# =========================
# 7. Quick diagnosis
# =========================

import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

# scratch 학습에 실제로 사용된 feature와 dataset을 그대로 사용
feature_columns = scratch_result["metrics"]["feature_columns"]
metrics = scratch_result["metrics"]
dataset_versions = metrics["dataset_versions"]

# feature importance 파일은 이미 importance 내림차순으로 저장되어 있음
fi = pd.read_csv(scratch_result["paths"]["importance"])

def load_datasets(versions):
    return pd.concat([
        pd.read_csv(PROJECT_ROOT / "data" / "processed" / f"features_{version}.csv", usecols=feature_columns + ["label"])
        for version in versions
    ], ignore_index=True)

if metrics["evaluation_strategy"] == "scenario_split_by_dataset_version":
    train_df = load_datasets(metrics["train_dataset_versions"])
    test_df = load_datasets(metrics["test_dataset_versions"])
    train_df, _ = sample_class_ratio(train_df, metrics["class_ratio"]["train"], split_name="train", random_seed=metrics["random_seed"])
    test_df, _ = sample_class_ratio(test_df, metrics["class_ratio"]["test"], split_name="test", random_seed=metrics["random_seed"] + 100)
    diag_df = pd.concat([train_df.assign(split="train"), test_df.assign(split="test")], ignore_index=True)
    X_train = train_df[feature_columns].apply(pd.to_numeric, errors="coerce")
    y_train = train_df["label"].astype("int8")
    X_test = test_df[feature_columns].apply(pd.to_numeric, errors="coerce")
    y_test = test_df["label"].astype("int8")
else:
    diag_df = load_datasets(dataset_versions)
    X = diag_df[feature_columns].apply(pd.to_numeric, errors="coerce")
    y = diag_df["label"].astype("int8")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

print("Rows:", len(diag_df))
print("Label distribution:")
print(diag_df["label"].value_counts().sort_index())
print("Evaluation strategy:", metrics["evaluation_strategy"])

print("\n[1] Top feature label-wise distribution")

for col in fi.head(10)["feature"]:
    print(f"\n===== {col} =====")
    display(diag_df.groupby("label")[col].describe())

print("\n[2] Single feature performance")

imputer = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_columns,
    index=X_train.index,
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_columns,
    index=X_test.index,
)

single_rows = []

for col in feature_columns:
    one_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=2,
        random_state=42,
        n_jobs=4,
        class_weight="balanced",
    )

    one_model.fit(X_train_imputed[[col]], y_train)
    pred = one_model.predict(X_test_imputed[[col]])

    single_rows.append({
        "feature": col,
        "f1": f1_score(y_test, pred, zero_division=0),
        "accuracy": accuracy_score(y_test, pred),
        "normal_mean": diag_df[diag_df["label"] == 0][col].mean(),
        "attack_mean": diag_df[diag_df["label"] == 1][col].mean(),
        "normal_unique": diag_df[diag_df["label"] == 0][col].nunique(),
        "attack_unique": diag_df[diag_df["label"] == 1][col].nunique(),
    })

single_check = pd.DataFrame(single_rows).sort_values(
    ["f1", "accuracy"],
    ascending=False,
)

display(single_check.head(30))

print("\n[3] Shallow decision tree rule")

tree = DecisionTreeClassifier(
    max_depth=2,
    random_state=42,
    class_weight="balanced",
)

tree.fit(X_train_imputed, y_train)
tree_pred = tree.predict(X_test_imputed)

print(classification_report(y_test, tree_pred, digits=4))
print(export_text(tree, feature_names=list(feature_columns)))

print("\n[4] Train/Test duplicate check")

train_rows = X_train_imputed.copy()
train_rows["_label"] = y_train.values
test_rows = X_test_imputed.copy()
test_rows["_label"] = y_test.values
common = train_rows.merge(test_rows, how="inner")
print("Train rows:", len(train_rows))
print("Test rows:", len(test_rows))
print("Exact duplicate rows across train/test:", len(common))
print("Duplicate ratio in test:", len(common) / len(test_rows))
display(common.head(20))

Rows: 54540
Label distribution:
label
0    23198
1    31342
Name: count, dtype: int64

[1] Top feature label-wise distribution

===== query_entropy =====


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,23198.0,3.900860,1.123913e-01,3.155565,3.855389,3.913800,3.962103,4.159200
1,31342.0,2.947703,4.440963e-16,2.947703,2.947703,2.947703,2.947703,2.947703



===== ip_len =====


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,23198.0,141.89081,17.068294,120.0,129.0,132.0,148.0,178.0
1,31342.0,109.00000,0.000000,109.0,109.0,109.0,109.0,109.0



===== frame_cap_len =====


...

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,23198.0,161.89081,17.068294,140.0,149.0,152.0,168.0,198.0
1,31342.0,129.00000,0.000000,129.0,129.0,129.0,129.0,129.0



[2] Single feature performance


,feature,f1,accuracy,normal_mean,attack_mean,normal_unique,attack_unique
0,frame_len,1.000000,1.000000,161.890810,129.000000,40,1
1,frame_cap_len,1.000000,1.000000,161.890810,129.000000,40,1
2,ip_len,1.000000,1.000000,141.890810,109.000000,40,1
5,udp_length,1.000000,1.000000,121.890810,89.000000,40,1
7,src_port,1.000000,1.000000,30053.000000,53.000000,1,1
33,ttl_min,1.000000,1.000000,354.927839,86400.000000,254,1
34,ttl_max,1.000000,1.000000,361.409259,86400.000000,254,1
35,ttl_mean,1.000000,1.000000,358.168549,86400.000000,447,1
37,query_name_len,1.000000,1.000000,28.392577,9.000000,14,1
38,query_label_count,1.000000,1.000000,3.847444,2.000000,2,1



[3] Shallow decision tree rule
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      4640
           1     1.0000    1.0000    1.0000      6268

    accuracy                         1.0000     10908
   macro avg     1.0000    1.0000    1.0000     10908
weighted avg     1.0000    1.0000    1.0000     10908

|--- ttl_mean <= 43800.00
|   |--- class: 0
|--- ttl_mean >  43800.00
|   |--- class: 1


[4] Train/Test duplicate check


NameError: name 'train' is not defined

In [ ]:
# Scratch 결과 상세 확인
display(pd.read_csv(scratch_result['paths']['confusion_matrix']))
display(pd.read_csv(scratch_result['paths']['importance']).head(20))